# AKE Lab 6 · Daten-Engineering: Trainingsdaten und Pipeline mit DVC — Starter-Notebook

**Angewandte KI & KI-Engineering · Block C – KI-Engineering & MLOps · Woche 6**
Bezug: Foliensatz *Daten-Engineering für ML*, Abschnitte 3 (Datenpipelines), 4 (Datenversionierung mit DVC), 6 (Bias in
Trainingsdaten) und 7 (Capstone) · Lektüre: Huyen (2022), *Designing Machine Learning Systems*, Kap. 3–4

> **Dieses Notebook ist das komplette Lab – es gibt kein separates Aufgabenblatt.** Jede Aufgabe enthält die Aufgabenstellung,
> die Befehle, die Erwartungswerte und die Leitfragen. Der Code ist ausführlich kommentiert: Lesen Sie die Kommentare
> (Zeilen, die mit `#` beginnen) – sie erklären Zeile für Zeile, was passiert.

**Die Aufgabe:** Nach Lab 2 gibt es ein gutes Trägermodell (98,4 % Validierungsgenauigkeit) – aber keinen Nachweis, aus welchen
Daten, mit welchem Code und welchen Parametern es entstand. Das ändern Sie heute: Sie legen ein eigenes Übungs-Repository
`traeger-pipeline/` an, stellen die Bilder unter **DVC** (Data Version Control), bauen aus den Schritten von Lab 2 eine
**Pipeline** mit vier Stufen und beobachten, was `dvc repro` nach einer Änderung neu rechnet. Danach prüfen Sie die Daten
auf Qualitätsprobleme und **Bias** und testen, ob Docker auf Ihrem Rechner läuft (Vorbereitung auf MLOps).

**Charakter:** Übung in **Blöcken während der Vorlesung** – jeder Block folgt direkt auf den Vorlesungsabschnitt, den er vertieft.
**Ihre Zeit ≈ 65 Min.** in der Sitzung plus 10 Min. Docker-Check in der Capstone-Sprechstunde, davon **≈ 3–6 Min. Rechenzeit**
(zwei Pipeline-Läufe; Referenz 4 Kerne / 8 GB, keine GPU).

## Organisatorisches

| | |
|---|---|
| **Termin** | Woche 6 (KW 46, 12.11.2026), drei Blöcke während der Vorlesung + Docker-Check in der Capstone-Sprechstunde; Besprechung jeweils direkt nach der Arbeitsphase am Beamer |
| **Bezug** | Foliensatz *Daten-Engineering für ML*, Abschn. 3 (Pipelines), 4 (DVC), 6 (Bias), 7 (Capstone) |
| **Material** | Ordner `lab06/` im Kursrepository: dieses Notebook und `vorlage/` (Pipeline: `dvc.yaml`, `params.yaml`, die vier Stufen-Skripte `lab06/prepare.py`, `featurize.py`, `train.py`, `evaluate.py`); die Musterlösung folgt nach der Sitzung |
| **Umgebung** | Kursumgebung aktualisieren: `git pull`, danach `uv sync`. DVC 3.67.1 ist schon enthalten. Zusätzlich nötig: **Git** (haben Sie seit Lab 1) und für Aufgabe 4 **Docker Desktop**. |
| **Daten** | `data/imagenette5/` (3 000 Bilder, 28,5 MB, wie in Lab 2), `data/models/resnet18-f37072fd.pth` (ResNet-18-Gewichte), `models/traeger_v1.yaml.template` |

**Fahrplan**

| Aufgabe | Block in der Vorlesung (nach Folie …) | Inhalt | Ihre Zeit | Rechenzeit |
|---|---|---|---|---|
| 0 · Setup | vorab, spätestens zu Beginn | Pfade, Hilfsfunktion `run()`, Prüfungen | 2 Min. | Sekunden |
| 1 · Daten mit DVC versionieren | „Duplikate: einmal geladen, zweimal im Ordner“ | Übungs-Repo, `dvc add` (`CODE`), Hash → Cache (`PRINZIP`), Remote (`CODE`), Bild hinzufügen (`CODE`) | 20 Min. | ≈ 10–20 s |
| 2 · Pipeline und `dvc repro` | „Zeitreise: einen alten Stand wiederherstellen“ | `dvc.yaml` lesen, erster Lauf, `dvc status` von Hand (`PRINZIP`, `CODE`), Parameter ändern (`CODE`), Run-Cache | 25 Min. | ≈ 3–6 Min. |
| 3 · Datenqualitäts- und Bias-Check | „Beispiel: Helligkeit und Farbe je Klasse“ | Datencheck, Duplikate von Hand (`PRINZIP`, `CODE`), Helligkeit je Klasse (`PRINZIP`, `CODE`), Bilder ansehen | 20 Min. | Sekunden |
| 4 · Docker-Installationscheck | Capstone-Sprechstunde | `docker --version`, `docker run hello-world`, Ursachen | 10 Min. | Sekunden |
| Zusatz | zu Hause | Zeitreise mit `git checkout` + `dvc checkout`; Kommentar in `train.py` | – | ≈ 1–2 Min. |

**Kennzeichnung der Arbeitsarten** (steht am Anfang jeder Teilaufgabe):

| Marke | Was Sie tun |
|---|---|
| `LIVE` | wird in der Vorlesung gemeinsam vorgeführt – ausführen und mitlesen |
| `CODE` | eine Lücke selbst füllen (Zeile mit `raise NotImplementedError` durch Ihre Lösung ersetzen) |
| `PRINZIP` | ausführen und lesen – bewusst einfacher Code, der zeigt, was DVC intern tut, und am Ende mit DVC vergleicht; oft mit einer kleinen Lücke |
| `AUSFÜHREN` | nur ausführen und die Ausgabe lesen |
| `DENKEN` | Interpretationsfrage in der Markdown-Zelle beantworten |

Zusätzlich steht dabei, ob die Teilaufgabe **gemeinsam** (navy auf der Lab-Folie) oder **allein** (orange) bearbeitet wird.
Wer bei einer Lücke nicht weiterkommt, übernimmt die Zeile aus der Lösung am Beamer – die Aufgaben bauen aufeinander auf.

**Abgabe** (Weg und Termin: siehe Lab-Briefing): das Übungs-Repo als ZIP (am Ende des Notebooks erzeugt – enthält nur, was in
Git liegt: Code, `.dvc`-Dateien, `dvc.yaml`, `dvc.lock`, `params.yaml`, Metriken) und dieses Notebook mit Ihren Antworten auf
die `DENKEN`-Fragen.

## Lernziele

Nach dem Lab können Sie …

- **Verstehen:** erklären, warum Git die Zeiger (`.dvc`, `dvc.lock`) und DVC die Daten versioniert und wie aus dem Inhalt einer
  Datei ihr Ablageort im Cache wird (inhaltsadressierter Speicher).
- **Anwenden:** einen Datensatz mit `dvc add` versionieren, über ein Remote teilen (`dvc push`/`pull`) und eine Pipeline mit
  `dvc.yaml` und `dvc repro` reproduzierbar ausführen.
- **Analysieren:** vorhersagen und begründen, welche Stufen nach einer Änderung neu laufen – und welche aus dem Run-Cache kommen.
- **Analysieren:** einen Bilddatensatz auf defekte Dateien, Duplikate und Bias (Helligkeit/Farbe je Klasse) prüfen.
- **Bewerten:** entscheiden, ob ein Unterschied zwischen Klassen eine legitime Eigenschaft oder ein Bias ist, und eine
  Gegenmaßnahme vorschlagen.

## Spielregeln und Hinweise

- **Eigenes Übungs-Repo.** Sie arbeiten nicht im Kursordner, sondern in `traeger-pipeline/` **daneben**. Würden Sie im
  Kursordner Dateien aus Git entfernen und committen, gäbe es beim nächsten `git pull` Konflikte.
- **Befehle laufen über `run("…")`.** Die Hilfsfunktion aus Aufgabe 0 führt einen Git- oder DVC-Befehl im Übungs-Repo aus und
  zeigt die Ausgabe so, wie Sie sie im Terminal sähen (`$ befehl` und darunter die Antwort). Wer lieber tippt: Im
  Jupyter-Terminal (*File → New → Terminal*) mit `cd` in den Ordner `traeger-pipeline` wechseln – die Befehle sind dieselben.
- **Immer von oben nach unten.** Git und DVC verändern Dateien auf der Festplatte. Eine Zelle zweimal auszuführen ist meist
  harmlos, kann aber eine Fehlermeldung geben („schon erledigt“). Lesen Sie die Meldung, bevor Sie etwas ändern.
- **Nach jedem `git push` auch `dvc push`.** Im Lab gibt es keinen Git-Server, also nur `dvc push` – aber die Regel gilt im Projekt.
- **Reproduzierbarkeit:** Die Stufen-Skripte setzen feste Seeds (`seed: 0` in `params.yaml`) und schreiben weder Datum noch
  Laufzeit in ihre Ausgaben – gleiche Eingaben ergeben so bitgleiche Ergebnisse.

## Kurzhilfe: Notebook, Python und die Befehle dieses Labs

**Bedienung:** Zelle auswählen und mit **Shift + Enter** ausführen. Nach einem Fehler die Zelle korrigieren und erneut ausführen.

| Sie sehen im Code … | Bedeutung |
|---|---|
| `# Text` | Kommentar – wird nicht ausgeführt, erklärt nur |
| `run("dvc status")` | führt den Befehl `dvc status` im Übungs-Repo aus und zeigt die Ausgabe |
| `run(["dvc", "remote", "add", "-d", "share", str(REMOTE)])` | derselbe Mechanismus, Befehl als Liste – nötig, wenn ein Pfad Leerzeichen enthalten kann |
| `ARBEIT / "data" / "imagenette5"` | Pfad zusammensetzen (`Path`-Objekt) – funktioniert unter Windows, macOS und Linux |
| `p.read_bytes()`, `p.read_text()` | Datei als Bytes bzw. als Text lesen |
| `hashlib.md5(daten).hexdigest()` | MD5-Prüfsumme als Text aus 32 Hexadezimalzeichen |
| `for x in liste:` | Schleife: Der eingerückte Block läuft für jedes Element |
| `d = {}`, `d["schluessel"] = wert`, `d.get(k, standard)` | *Dictionary* (Schlüssel → Wert) |
| `f"{x:.1f}"` | *f-String*: setzt `x` mit einer Nachkommastelle in den Text ein |
| `check("Name", lambda: ...)` | Prüfzelle: zeigt ✅, wenn die Bedingung hinter `lambda:` wahr ist, sonst ❌ mit Hinweis |
| `raise NotImplementedError("TODO …")` | Platzhalter in einer Lücke: Diese Zeile ersetzen Sie durch Ihre Lösung |

| Befehl | Was er tut |
|---|---|
| `git add <datei>` · `git commit -m "…"` | Datei für den nächsten Stand vormerken · Stand festhalten |
| `git rm -r --cached <ordner>` | Ordner aus Git entfernen, **Dateien bleiben auf der Festplatte** |
| `git log --oneline` · `git diff` | Liste der Stände · Unterschiede seit dem letzten Stand |
| `dvc init` | DVC im Git-Repo einrichten (Ordner `.dvc/`) |
| `dvc add <ordner>` | Daten in den Cache legen, Zeiger `<ordner>.dvc` schreiben, Ordner in `.gitignore` eintragen |
| `dvc remote add -d <name> <pfad>` | Standard-Remote festlegen |
| `dvc push` · `dvc pull` | fehlende Dateien zum Remote schicken · vom Remote holen und auspacken |
| `dvc status` | Passen Arbeitsordner und Zeiger (`.dvc`, `dvc.lock`) zusammen? |
| `dvc dag` · `dvc repro` | Pipeline-Graph zeigen · Pipeline ausführen, nur geänderte Stufen |
| `dvc params diff` · `dvc metrics show` · `dvc metrics diff` | Parameter bzw. Metriken anzeigen und zwischen Ständen vergleichen |
| `dvc checkout` | Daten passend zu den Zeigern in den Arbeitsordner holen |

## Aufgabe 0 · Setup

`AUSFÜHREN` · Bibliotheken laden, Pfade festlegen, die Hilfsfunktion `run()` bereitstellen und prüfen, ob Git, DVC und die
Daten da sind. Wenn alle Zeilen ✅ zeigen, kann es losgehen.

In [ ]:
# --- Bibliotheken -------------------------------------------------------------------------------------------
import os, re, sys, time, json, shutil, shlex, subprocess, hashlib, types, stat   # Standardbibliothek
from pathlib import Path                         # Pfade zu Dateien und Ordnern, unabhaengig vom Betriebssystem
import numpy as np                               # Rechnen mit Vektoren und Matrizen
import pandas as pd                              # Tabellen (DataFrame)
import matplotlib.pyplot as plt                  # Diagramme
import yaml                                      # YAML-Dateien lesen und schreiben (dvc.yaml, params.yaml, dvc.lock)
from PIL import Image                            # Bilder oeffnen (Aufgabe 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

# --- Pruefhelfer --------------------------------------------------------------------------------------------
# check("Name", Bedingung, "Hinweis") gibt ✅ aus, wenn die Bedingung wahr ist, sonst ❌ mit Hinweis.
# Die Bedingung steht meist als `lambda: ...` da: eine kleine Funktion, die erst hier ausgewertet wird.
CHECKS = {}
def check(name, ok, hint=""):
    try:
        ok = bool(ok() if callable(ok) else ok); msg = hint
    except NotImplementedError as e:             # die Luecke ist noch nicht gefuellt
        ok, msg = False, f"{e} ist noch offen"
    except Exception as e:                       # irgendein anderer Fehler
        ok, msg = False, f"{type(e).__name__}: {e}" + (f" | {hint}" if hint else "")
    CHECKS[name] = ok
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {msg}")

def nah(a, b, tol):                              # |a - b| <= tol  (fuer Kontrollwerte mit Rundung)
    return abs(float(a) - float(b)) <= tol

# --- Laufzeitmessung je Aufgabe (Tabelle am Ende; fuer das Lab nicht wichtig) ---------------------------------
RUNTIME, _teil = {}, {"name": "Aufgabe 0", "t0": None}
def abschnitt(name): _teil["name"] = name
def _ake_pre(*a): _teil["t0"] = time.perf_counter()
def _ake_post(*a):
    if _teil["t0"] is not None:
        RUNTIME[_teil["name"]] = RUNTIME.get(_teil["name"], 0.0) + time.perf_counter() - _teil["t0"]
try:
    _ip = get_ipython()
    for ev, fn in (("pre_run_cell", _ake_pre), ("post_run_cell", _ake_post)):
        for old in [f for f in _ip.events.callbacks[ev] if getattr(f, "__name__", "") == fn.__name__]:
            _ip.events.unregister(ev, old)
        _ip.events.register(ev, fn)
except NameError:
    pass

# --- Pfade ----------------------------------------------------------------------------------------------------
# REPO_ROOT = Kursordner (erster Elternordner mit pyproject.toml). Das Uebungs-Repo ARBEIT und das Remote REMOTE liegen
# DANEBEN. Wer andere Orte will (z. B. Hochschul-Share als Remote), setzt hier einen anderen Pfad ein.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden. Bitte Jupyter im Ordner des Repositorys starten und lab06/ oeffnen.")
DATA    = REPO_ROOT / "data"
VORLAGE = REPO_ROOT / "lab06" / "vorlage"                     # dvc.yaml, params.yaml, lab06/*.py
ARBEIT  = Path(os.environ.get("AKE_LAB06_ARBEIT", REPO_ROOT.parent / "traeger-pipeline")).resolve()   # Uebungs-Repo
REMOTE  = Path(os.environ.get("AKE_LAB06_REMOTE", REPO_ROOT.parent / "dvc-remote")).resolve()         # DVC-Remote
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

# --- run(): einen Git- oder DVC-Befehl im Uebungs-Repo ausfuehren ----------------------------------------------
# Was die Funktion tut:
#   1. Den Befehl in seine Teile zerlegen ("dvc add data/x" -> ["dvc", "add", "data/x"]).
#   2. "dvc" durch "<diese Python-Umgebung> -m dvc" ersetzen - so laeuft garantiert das DVC der Kursumgebung.
#   3. Den Ordner der Kursumgebung vorne in den Suchpfad (PATH) setzen - so findet `dvc repro` das richtige `python`.
#   4. Den Befehl im Ordner ARBEIT starten und jede Ausgabezeile sofort anzeigen (wie im Terminal).
# Rueckgabe: Objekt mit .ok (True/False), .ausgabe (Text) und .sekunden (Laufzeit).
RAUSCHEN = ("To enable auto staging", "dvc config core.autostage true")   # Hinweiszeilen, die nur ablenken
SPINNER = set("⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏")                                       # Zeichen der DVC-Fortschrittsanzeige
LAUFZEITEN = {}                                                           # Laufzeit je Befehl (fuer die Tabelle am Ende)
def run(befehl, erwarte_fehler=False, zeigen=True, cwd=None):
    teile = shlex.split(befehl) if isinstance(befehl, str) else [str(t) for t in befehl]
    text = befehl if isinstance(befehl, str) else " ".join(teile)
    if teile[0] == "dvc":
        teile = [sys.executable, "-m", "dvc"] + teile[1:]
    env = dict(os.environ, PATH=str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""),
               PYTHONUNBUFFERED="1", DVC_NO_ANALYTICS="1", GIT_PAGER="cat", PAGER="cat", TERM="dumb")
    if zeigen:
        print(f"$ {text}", flush=True)
    t0 = time.perf_counter()
    try:
        p = subprocess.Popen(teile, cwd=cwd or ARBEIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, encoding="utf-8", errors="replace", env=env)
    except FileNotFoundError:
        print(f"❌ Programm '{teile[0]}' nicht gefunden."); return types.SimpleNamespace(ok=False, ausgabe="", sekunden=0)
    zeilen = []
    for zeile in p.stdout:                                  # Zeile fuer Zeile, waehrend der Befehl laeuft
        zeile = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", zeile).rstrip()   # Steuerzeichen fuer Farben/Fortschrittsanzeigen entfernen
        if any(r in zeile for r in RAUSCHEN) or zeile.lstrip()[:1] in SPINNER or (not zeile and (not zeilen or not zeilen[-1])):
            continue                                        # Hinweis-, Fortschritts- und doppelte Leerzeilen weglassen
        zeilen.append(zeile)
        if zeigen:
            print("   " + zeile, flush=True)
    p.wait()
    erg = types.SimpleNamespace(ok=(p.returncode == 0), ausgabe="\n".join(zeilen), sekunden=time.perf_counter() - t0)
    LAUFZEITEN[f"{len(LAUFZEITEN) + 1:3d}: {text}"[:70]] = erg.sekunden   # laufende Nummer, damit Wiederholungen nicht ueberschrieben werden
    if not erg.ok and not erwarte_fehler:
        raise RuntimeError(f"Befehl fehlgeschlagen (Rueckgabewert {p.returncode}): {text} – Ausgabe oben lesen.")
    if not erg.ok and zeigen:
        print("   (Fehler war hier erwartet – Meldung lesen!)")
    return erg

def loesche(ordner):
    """Ordner samt Inhalt loeschen - auch schreibgeschuetzte Dateien (DVC schuetzt seinen Cache, Windows verweigert sonst)."""
    def _schreibbar_machen(func, pfad, exc):
        os.chmod(pfad, stat.S_IWRITE); func(pfad)
    shutil.rmtree(ordner, onexc=_schreibbar_machen)

# --- Pruefungen -----------------------------------------------------------------------------------------------
v_dvc = run("dvc --version", cwd=REPO_ROOT, zeigen=False)
v_git = run("git --version", cwd=REPO_ROOT, zeigen=False, erwarte_fehler=True)
print(f"Kursordner:  {REPO_ROOT}\nUebungs-Repo: {ARBEIT}\nRemote:      {REMOTE}\nDVC {v_dvc.ausgabe.strip()} | {v_git.ausgabe.strip()}")
check("0 DVC 3.x in der Kursumgebung", lambda: v_dvc.ausgabe.strip().startswith("3."), "uv sync ausfuehren")
check("0 Git installiert", v_git.ok, "Git installieren (git-scm.com) und Jupyter neu starten")
check("0 Daten da: data/imagenette5 mit train.txt, val.txt, classes.json",
      lambda: all((DATA / "imagenette5" / f).exists() for f in ("train.txt", "val.txt", "classes.json")), "git pull im Kursordner")
check("0 ResNet-18-Gewichte und Modellvorlage da",
      lambda: (DATA / "models" / "resnet18-f37072fd.pth").exists() and (REPO_ROOT / "models" / "traeger_v1.yaml.template").exists())
check("0 Vorlage der Pipeline da (lab06/vorlage)", lambda: (VORLAGE / "dvc.yaml").exists(), "git pull im Kursordner")

## Aufgabe 1 · Daten mit DVC versionieren

**Block nach „Duplikate: einmal geladen, zweimal im Ordner“ · 20 Min. (11 allein + 9 gemeinsam)** · Folien Abschnitt 4 ·
doc.dvc.org (DVC 3.67.1)

Sie stellen das Trägermodell-Repo unter DVC und schicken die Daten einmal über das Remote: Git bekommt nur einen 114 Byte
kleinen Zeiger, DVC die 28,5 MB Bilder.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 1.1 | Übungs-Repo anlegen | `LIVE` · gemeinsam | 4 Min. |
| 1.2 | Daten unter DVC + Hash → Cache von Hand | `LIVE` + `CODE` + `PRINZIP` · gemeinsam | 5 Min. |
| 1.3 | Remote und Rundreise | `CODE` · allein | 6 Min. |
| 1.4 | Ein Bild hinzufügen | `CODE` · allein | 5 Min. |

### 1.1 · Übungs-Repo anlegen

`LIVE` · gemeinsam · Die Zelle kopiert den **Stand nach Lab 2** in den neuen Ordner `traeger-pipeline/` – mit denselben
relativen Pfaden wie im Kursordner –, legt dort ein Git-Repo an und hält alles in einem ersten Commit fest.

```
traeger-pipeline/
├── data/imagenette5/            3 000 Bilder + train.txt, val.txt, classes.json
├── data/models/resnet18-f37072fd.pth
├── models/traeger_v1.yaml.template
└── lab02/src/traeger.py         Datensatz, Vorverarbeitung, Modell (aus Lab 2)
```

**Erwartung:** ein Commit „Stand Lab 2“; die Bilder liegen zunächst **in Git** – genau das ändern wir in 1.2.

In [ ]:
abschnitt("Aufgabe 1")
NEU_ANFANGEN = False        # True: Uebungs-Repo und Remote loeschen und neu anlegen (nur Ordner, die dieses Notebook angelegt hat)

MARKE = ".ake_lab06"        # Markierungsdatei: zeigt, dass ein Ordner von diesem Notebook stammt
if NEU_ANFANGEN:
    for ordner in (ARBEIT, REMOTE):
        if (ordner / MARKE).exists():
            loesche(ordner); print("geloescht:", ordner)
        elif ordner.exists():
            print(f"⚠️  {ordner} stammt nicht von diesem Notebook – bitte von Hand pruefen und loeschen.")

if (ARBEIT / ".git").exists():
    print(f"Uebungs-Repo existiert schon: {ARBEIT} – wird weiterverwendet (neu anfangen: NEU_ANFANGEN = True).")
else:
    # (1) Stand nach Lab 2 kopieren - gleiche relative Pfade wie im Kursordner
    ARBEIT.mkdir(parents=True)
    (ARBEIT / MARKE).write_text("angelegt von lab06\n")
    shutil.copytree(DATA / "imagenette5", ARBEIT / "data" / "imagenette5")
    (ARBEIT / "data" / "models").mkdir()
    shutil.copy2(DATA / "models" / "resnet18-f37072fd.pth", ARBEIT / "data" / "models")
    (ARBEIT / "models").mkdir()
    shutil.copy2(REPO_ROOT / "models" / "traeger_v1.yaml.template", ARBEIT / "models")
    shutil.copytree(REPO_ROOT / "lab02" / "src", ARBEIT / "lab02" / "src", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copy2(VORLAGE / ".gitignore", ARBEIT / ".gitignore")          # __pycache__ usw. nicht versionieren
    # (2) Git-Repo anlegen, Hauptzweig "main"
    run("git init")
    run("git symbolic-ref HEAD refs/heads/main")
    # (3) Git braucht fuer Commits Name und E-Mail. Fehlen sie, nur fuer DIESES Repo Platzhalter setzen.
    if not run("git config user.email", erwarte_fehler=True, zeigen=False).ausgabe.strip():
        run(["git", "config", "user.name", "AKE Studi"]); run(["git", "config", "user.email", "studi@example.invalid"])
        print("Git-Identitaet gesetzt (nur fuer dieses Uebungs-Repo).")
    # (4) alles committen
    run("git add .")
    run('git commit -q -m "Stand Lab 2"')
run("git log --oneline")
check("1.1 Uebungs-Repo mit Commit 'Stand Lab 2'", lambda: "Stand Lab 2" in run("git log --oneline", zeigen=False).ausgabe)
check("1.1 3 003 Dateien in data/imagenette5", lambda: sum(1 for p in (ARBEIT / "data/imagenette5").rglob("*") if p.is_file()) == 3003)

### 1.2 · Daten unter DVC

`LIVE` + `CODE` (2 Zeilen) · gemeinsam · Drei Schritte:

1. `dvc init` richtet DVC ein (Ordner `.dvc/` mit Konfiguration und später dem Cache) – vorgegeben.
2. Erst ein **Fehlversuch**: `dvc add data/imagenette5`, während Git den Ordner noch verwaltet. Lesen Sie die Meldung –
   sie sagt selbst, was zu tun ist.
3. `CODE`: den Ordner mit `git rm -r --cached` aus Git nehmen (die Dateien bleiben auf der Festplatte!) und dann mit
   `dvc add` an DVC übergeben.

**Erwartung:** `data/imagenette5.dvc` mit `md5: fb66058b….dir`, `size: 28531866`, `nfiles: 3003`; in `data/.gitignore`
steht `/imagenette5`.

In [ ]:
run("dvc init -q")                                    # legt .dvc/ an und merkt die neuen Dateien in Git vor
run('git commit -q -m "DVC eingerichtet"', erwarte_fehler=True)   # erwarte_fehler: beim zweiten Ausfuehren gibt es nichts zu committen
print()
run("dvc add data/imagenette5", erwarte_fehler=True)  # Fehlversuch: Git verwaltet den Ordner noch

In [ ]:
# TODO 1.2: 
# - zunächst den imagenette5-Ordner aus dem Git-cache löschen
# - anschließend den imagenette5-ordner zu dvc hinzufügen
raise NotImplementedError("TODO 1.2")
print("\n--- data/imagenette5.dvc (das ist alles, was Git kuenftig sieht):")
print((ARBEIT / "data/imagenette5.dvc").read_text())
print("--- data/.gitignore:", (ARBEIT / "data/.gitignore").read_text())
run("git add data/imagenette5.dvc data/.gitignore")
run('git commit -q -m "Bilder unter DVC"')

zeiger = yaml.safe_load((ARBEIT / "data/imagenette5.dvc").read_text())["outs"][0]
check("1.2 .dvc-Datei: md5 fb66058b...dir, 3 003 Dateien, 28 531 866 Byte",
      lambda: zeiger["md5"].startswith("fb66058b") and zeiger["nfiles"] == 3003 and zeiger["size"] == 28531866)
check("1.2 Git verwaltet die Bilder nicht mehr",
      lambda: "imagenette5/train.txt" not in run("git ls-files data", zeigen=False).ausgabe)
check(f"1.2 Zeiger ist klein: {(ARBEIT / 'data/imagenette5.dvc').stat().st_size} Byte",
      lambda: (ARBEIT / "data/imagenette5.dvc").stat().st_size < 200)

#### 1.2 `PRINZIP` · Vom Inhalt zum Ablageort – von Hand nachgerechnet

`PRINZIP` + `CODE` (1 Zeile) · Folie „Vom Inhalt zum Ablageort“: DVC benennt jede Datei nach der **MD5-Prüfsumme ihres
Inhalts**. Die ersten 2 Zeichen werden der Ordner, die restlichen 30 der Dateiname:

$$\texttt{.dvc/cache/files/md5/}\;\underbrace{\texttt{ec}}_{\text{2 Zeichen}}\texttt{/}\;\underbrace{\texttt{766f1bb7…055c}}_{\text{30 Zeichen}}$$

Für einen Ordner legt DVC zusätzlich eine **Inhaltsliste** ab (`.dir`-Datei): alle Dateien, alphabetisch nach Pfad, je mit
`md5` und `relpath`, als JSON-Text. Die Prüfsumme dieses Textes ist die `md5` in der `.dvc`-Datei.

Wir rechnen beides mit einfachem Python nach – ohne DVC – und vergleichen am Ende mit dem, was DVC geschrieben hat.
**Erwartung:** Das Kirchenbild von der Folie hat `ec766f1b…`; die selbst berechnete Ordner-Prüfsumme ist `fb66058b…`.

In [ ]:
# Hilfsfunktion fuer (b) - Ordner-Pruefsumme wie bei DVC; wird in Aufgabe 2 und 3 wiederverwendet
def md5_ordner(ordner, zeigen=False):
    """Pruefsumme eines Ordners wie bei DVC: md5 der JSON-Liste aller Dateien (sortiert nach relativem Pfad)."""
    liste = []
    for p in sorted(Path(ordner).rglob("*")):
        if p.is_file():
            rel = p.relative_to(ordner).as_posix()                 # immer mit "/" - auch unter Windows
            liste.append({"md5": hashlib.md5(p.read_bytes()).hexdigest(), "relpath": rel})
    liste.sort(key=lambda e: e["relpath"])                         # alphabetisch nach Pfad (Text-Sortierung)
    text = json.dumps(liste, sort_keys=True)                       # so schreibt DVC die .dir-Datei
    if zeigen:
        print(f"{len(liste)} Eintraege, die ersten zwei:", json.dumps(liste[:2], sort_keys=True))
    return hashlib.md5(text.encode()).hexdigest()

# (a) Eine Datei: Pruefsumme -> Pfad im Cache
bild = ARBEIT / "data/imagenette5/train/n03028079/ILSVRC2012_val_00008116.JPEG"   # Kirche, 9 583 Byte (Folie)
md5 = hashlib.md5()
with open(bild, "rb") as f:                     # Datei in Stuecken lesen - so ginge es auch mit 10-GB-Dateien
    while True:
        stueck = f.read(1024 * 1024)            # 1 MB
        if not stueck:                          # leer = Dateiende
            break
        md5.update(stueck)                      # Pruefsumme fortschreiben
h = md5.hexdigest()
print(f"Datei:     {bild.name} ({bild.stat().st_size} Byte)")
print(f"md5:       {h}")
# TODO 1.2P: Pfad im Cache: ARBEIT / ".dvc/cache/files/md5" / (erste 2 Zeichen von h) / (restliche 30 Zeichen von h)
raise NotImplementedError("TODO 1.2P")
print(f"Cache:     {cache_pfad.relative_to(ARBEIT)}")
print(f"existiert: {cache_pfad.exists()} | gleicher Inhalt: {cache_pfad.exists() and cache_pfad.read_bytes() == bild.read_bytes()}")

# (b) Ein Ordner: Inhaltsliste bauen und ihre Pruefsumme mit md5_ordner() (oben) berechnen
h_ordner = md5_ordner(ARBEIT / "data/imagenette5", zeigen=True)
print(f"selbst berechnet: {h_ordner}.dir")
print(f"laut .dvc-Datei:  {zeiger['md5']}")
dir_datei = ARBEIT / ".dvc/cache/files/md5" / h_ordner[:2] / (h_ordner[2:] + ".dir")
print("Anfang der .dir-Datei im Cache:", dir_datei.read_text()[:110], "...")

check("1.2P Kirchenbild hat md5 ec766f1b...", h.startswith("ec766f1b"))
check("1.2P Cache-Pfad richtig und Datei liegt dort", lambda: cache_pfad.exists() and cache_pfad.read_bytes() == bild.read_bytes(),
      "Ordner = h[:2], Dateiname = h[2:]")
check("1.2P Ordner-Pruefsumme von Hand == DVC", lambda: zeiger["md5"] == h_ordner + ".dir")

### 1.3 · Remote und Rundreise

`CODE` (2 Zeilen) · allein · Der Cache liegt nur auf Ihrem Rechner. Ein **Remote** ist der gemeinsame Speicher fürs Team –
hier ein Ordner `dvc-remote/` neben dem Kursordner (im Projekt ein Share, S3 o. Ä.).

1. `CODE`: Remote als Standard eintragen: `run(["dvc", "remote", "add", "-d", "share", str(REMOTE)])` – als Liste, weil der Pfad
   Leerzeichen enthalten kann. Die Einstellung landet in `.dvc/config` und wird committet (vorgegeben).
2. `CODE`: `dvc push` – lädt alles hoch, was im Remote fehlt.
3. Rundreise (vorgegeben): Cache **und** Bildordner löschen, `dvc status` lesen, `dvc pull` – alles ist wieder da.

**Erwartung:** „3004 files pushed“ (3 003 Dateien + 1 `.dir`-Liste), im Remote ≈ 33 MB; nach dem Löschen meldet `dvc status`
„not in cache“, `dvc pull` „3004 files fetched and 3003 files added“.

In [ ]:
REMOTE.mkdir(parents=True, exist_ok=True); (REMOTE / MARKE).write_text("angelegt von lab06\n")
if "share" not in run("dvc remote list", zeigen=False).ausgabe:
    # TODO 1.3a: Remote "share" als Standard (-d) eintragen: 
    raise NotImplementedError("TODO 1.3a")
    run("git add .dvc/config")
    run('git commit -q -m "Remote eingetragen"')
print((ARBEIT / ".dvc/config").read_text())
# TODO 1.3b: alles hochladen, was im Remote fehlt: 
raise NotImplementedError("TODO 1.3b")

def dateien_und_mb(ordner):
    """Anzahl Dateien und Groesse in MB (ohne unsere Markierungsdatei)."""
    fs = [p for p in Path(ordner).rglob("*") if p.is_file() and p.name != MARKE]
    return len(fs), sum(p.stat().st_size for p in fs) / 1e6
print("Remote: %d Dateien, %.1f MB | Cache: %d Dateien, %.1f MB"
      % (*dateien_und_mb(REMOTE / "files"), *dateien_und_mb(ARBEIT / ".dvc/cache/files")))
check("1.3 Remote enthaelt 3 004 Objekte (3 003 Dateien + 1 .dir)", lambda: dateien_und_mb(REMOTE / "files")[0] >= 3004)

In [ ]:
# Rundreise: so, als waere der Rechner neu - Cache und Bilder weg, nur die Git-Dateien sind noch da
loesche(ARBEIT / ".dvc/cache")
loesche(ARBEIT / "data/imagenette5")
print("Cache und data/imagenette5 geloescht.\n")
run("dvc status")                                    # was fehlt?
erg_pull = run("dvc pull")                           # vom Remote holen und auspacken
check("1.3 nach dvc pull wieder 3 003 Dateien da",
      lambda: sum(1 for p in (ARBEIT / "data/imagenette5").rglob("*") if p.is_file()) == 3003)
check("1.3 Arbeitsordner passt zum Zeiger (md5 von Hand)", lambda: md5_ordner(ARBEIT / "data/imagenette5") + ".dir" == zeiger["md5"])

### 1.4 · Ein Bild hinzufügen

`CODE` (2 Zeilen) · allein · Ein Datensatz ändert sich: Sie kopieren ein Validierungsbild der Klasse *Kirche* unter neuem
Namen und tragen es in `val.txt` ein (vorgegeben). Dann:

1. `dvc status` lesen – was meldet DVC?
2. `CODE`: `dvc add data/imagenette5` – neue Version in den Cache, Zeiger aktualisieren.
3. `git diff` auf die `.dvc`-Datei – was ändert sich am Zeiger? Danach committen (vorgegeben).
4. `CODE`: `dvc push` – wie viele Dateien werden übertragen?

**Erwartung:** `dvc status` meldet `modified: data/imagenette5`; im `git diff` ändern sich `md5` und `size`, `nfiles` wird 3004;
`dvc push` überträgt **2 Dateien**.

In [ ]:
img = ARBEIT / "data/imagenette5"
quelle = "val/n03028079/ILSVRC2012_val_00004912.JPEG"         # ein Validierungsbild der Klasse Kirche (Label 1)
kopie = "val/n03028079/kopie_00004912.JPEG"
if not (img / kopie).exists():
    shutil.copy2(img / quelle, img / kopie)                   # Bild unter neuem Namen kopieren
    with open(img / "val.txt", "a", newline="\n") as f:       # Zeile anhaengen: Pfad<TAB>Label
        f.write(f"{kopie}\t1\n")
print("val.txt endet mit:", (img / "val.txt").read_text().splitlines()[-1])
run("dvc status")
# TODO 1.4a: neue Version an DVC uebergeben: 
raise NotImplementedError("TODO 1.4a")
run("git diff data/imagenette5.dvc")
run("git add data/imagenette5.dvc")
run('git commit -q -m "Kirche: Kopie eines Validierungsbildes"', erwarte_fehler=True)
# TODO 1.4b: hochladen: erg_push2 = ??
raise NotImplementedError("TODO 1.4b")
zeiger2 = yaml.safe_load((ARBEIT / "data/imagenette5.dvc").read_text())["outs"][0]
check(f"1.4 nfiles 3 004 (gemessen {zeiger2['nfiles']}), neue md5 {zeiger2['md5'][:8]}...",
      lambda: zeiger2["nfiles"] == 3004 and zeiger2["md5"] != zeiger["md5"])
check("1.4 dvc push hat 2 Dateien uebertragen", lambda: "2 files pushed" in erg_push2.ausgabe,
      "schon einmal gepusht? Dann steht dort 'Everything is up to date'")

**`DENKEN` – Interpretation**

a) Warum ändert sich in 1.4 die `md5` des **Ordners**, obwohl die 3.003 alten Dateien gleich geblieben sind?

b) Was genau liegt nach 1.3 im Remote ==> und warum ist es fast genauso groß wie der Cache?

c) `dvc push` hat in 1.4 **zwei** Dateien übertragen, obwohl Sie ein Bild hinzugefügt haben. Welche zwei? Und warum nicht das Bild?

d) Was wäre in 1.3 passiert, wenn Sie vor dem Löschen **nicht** gepusht hätten?

## Aufgabe 2 · Pipeline und `dvc repro`

**Block nach „Zeitreise: einen alten Stand wiederherstellen“ · 25 Min. (13 allein + 12 gemeinsam)** · Folien Abschnitt 4,
Beispiel „Was läuft nach einer Änderung neu?“

Die Schritte aus Lab 2 werden vier Skripte mit deklarierten Ein- und Ausgaben. DVC führt sie aus und merkt sich in
`dvc.lock` die Prüfsummen aller Eingaben und Ausgaben. Danach beobachten Sie, was nach einer Änderung neu läuft.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 2.1 | `dvc.yaml` lesen, `dvc dag` | `AUSFÜHREN` · gemeinsam | 5 Min. |
| 2.2 | Erster Lauf, `dvc.lock`, `dvc status` von Hand | `LIVE` + `PRINZIP` + `CODE` · gemeinsam | 7 Min. |
| 2.3 | Parameter ändern | `CODE` + `DENKEN` · allein | 7 Min. |
| 2.4 | Vergleichen und zurück (Run-Cache) | `CODE` · allein | 6 Min. |

### 2.1 · `dvc.yaml` lesen

`AUSFÜHREN` · gemeinsam · Die Zelle kopiert die Pipeline aus `lab06/vorlage/` ins Übungs-Repo, zeigt `dvc.yaml` und
`params.yaml` und zeichnet mit `dvc dag` den Graphen.

| Stufe | Skript | liest | schreibt | aus Lab 2 |
|---|---|---|---|---|
| prepare | `lab06/prepare.py` | `data/imagenette5` | `lab06/data/manifest.csv` (Pfad, Label, md5 je Bild), `lab06/reports/datencheck.json` | Aufg. 0 |
| featurize | `lab06/featurize.py` | Manifest, ResNet-18-Gewichte | `lab06/cache/features_resnet18_160_{train,val}.pt` | Aufg. 4.2 |
| train | `lab06/train.py` | Manifest, Features, Gewichte, `params.yaml` → `train` | `models/traeger_v1.pt`, `models/traeger_v1.yaml` | Aufg. 4.3–4.5 |
| evaluate | `lab06/evaluate.py` | Modell, Manifest | `lab06/reports/metrics.json` | Aufg. 5 |

Lesen Sie in `dvc.yaml` für jede Stufe: Was steht unter `deps`? Warum steht dort auch das Skript selbst und `lab02/src/traeger.py`?

In [ ]:
abschnitt("Aufgabe 2")
# Pipeline-Dateien aus der Vorlage kopieren (dvc.yaml, params.yaml, lab06/*.py)
for quelle in VORLAGE.rglob("*"):
    if quelle.is_file() and quelle.name != ".gitignore" and "__pycache__" not in quelle.parts:
        ziel = ARBEIT / quelle.relative_to(VORLAGE)
        if not ziel.exists():                                  # vorhandene (evtl. geaenderte) Dateien nicht ueberschreiben
            ziel.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(quelle, ziel)
print("=== dvc.yaml ==========================================")
print((ARBEIT / "dvc.yaml").read_text())
print("=== params.yaml =======================================")
print((ARBEIT / "params.yaml").read_text())
run("dvc dag")
run("git add dvc.yaml params.yaml lab06")
run('git commit -q -m "Pipeline: dvc.yaml, params.yaml, Stufen-Skripte"', erwarte_fehler=True)
stufen = yaml.safe_load((ARBEIT / "dvc.yaml").read_text())["stages"]
check("2.1 vier Stufen prepare -> featurize -> train -> evaluate", list(stufen) == ["prepare", "featurize", "train", "evaluate"])

**`DENKEN` (1 Min.):** `dvc dag` zeigt, dass `train` mehrere Pfeile bekommt. Welche Eingaben hat `train`, und warum muss
`featurize` vorher laufen, obwohl das nirgends als Reihenfolge steht?

### 2.2 · Erster Lauf

`LIVE` · gemeinsam · `dvc repro` führt alle vier Stufen aus. Jedes Skript gibt seine Laufzeit aus – notieren Sie sie. Danach
öffnen wir `dvc.lock`, committen und pushen.

**Erwartung:** Val-Genauigkeit **98,4 %** – derselbe Wert wie in Lab 2; Laufzeit ≈ 2 Min. auf schnellen Rechnern, bis ≈ 4 Min. auf
der Referenz (featurize ≈ 25–50 s, train ≈ 70–140 s). Während es läuft: Ausgabe mitlesen („Running stage …“).

In [ ]:
erg_repro1 = run("dvc repro")
print("\n=== dvc.lock, Stufe train (Ausschnitt) ===")
lock = yaml.safe_load((ARBEIT / "dvc.lock").read_text())
print(yaml.safe_dump({"train": lock["stages"]["train"]}, sort_keys=False)[:1200])
run("git add dvc.lock lab06/reports lab06/data/.gitignore lab06/cache/.gitignore models/.gitignore")
run('git commit -q -m "Pipeline v1: erster Lauf"', erwarte_fehler=True)
V1 = run("git rev-parse --short HEAD", zeigen=False).ausgabe.strip()      # Commit merken (fuer die Zusatzaufgabe)
run("dvc push")
metriken1 = json.loads((ARBEIT / "lab06/reports/metrics.json").read_text())
print(f"\nmetrics.json: {metriken1}")
check("2.2 alle vier Stufen gelaufen", lambda: all(f"Running stage '{s}'" in erg_repro1.ausgabe for s in stufen),
      "schon einmal gelaufen? Dann steht 'didn't change, skipping' da - auch gut")
check(f"2.2 Val-Genauigkeit {metriken1['accuracy']:.1%} >= 95 % und gleich wie beim Training",
      lambda: metriken1["accuracy"] >= 0.95 and metriken1["gleich_wie_train"])

#### 2.2 `PRINZIP` · `dvc status` von Hand

`PRINZIP` + `CODE` (1 Zeile) · Woher weiß DVC, welche Stufe neu laufen muss? Für jede Stufe steht in `dvc.lock` die Prüfsumme
jeder Eingabe (`deps`) und der Wert jedes Parameters (`params`) **vom letzten Lauf**. DVC berechnet die aktuellen Prüfsummen
und vergleicht:

$$\text{Stufe veraltet} \iff \exists\ \text{Eingabe } d:\ \operatorname{md5}_\text{jetzt}(d) \neq \operatorname{md5}_\text{dvc.lock}(d)\ \ \lor\ \ \exists\ \text{Parameter } k:\ \text{Wert}_\text{jetzt}(k) \neq \text{Wert}_\text{dvc.lock}(k)$$

Für Ordner nehmen wir `md5_ordner()` aus 1.2. Die Funktion `status_von_hand()` brauchen wir in 2.3 wieder.
**Erwartung:** direkt nach dem Lauf sind alle vier Stufen aktuell – wie bei `dvc status` („Data and pipelines are up to date“).

In [ ]:
def md5_von(pfad):
    """Pruefsumme wie in dvc.lock: Datei -> md5 des Inhalts, Ordner -> md5 der Inhaltsliste + '.dir'."""
    pfad = ARBEIT / pfad
    return md5_ordner(pfad) + ".dir" if pfad.is_dir() else hashlib.md5(pfad.read_bytes()).hexdigest()

def status_von_hand():
    lock = yaml.safe_load((ARBEIT / "dvc.lock").read_text())        # Stand vom letzten Lauf
    params = yaml.safe_load((ARBEIT / "params.yaml").read_text())   # Stand jetzt
    veraltet = []
    for name, stufe in lock["stages"].items():
        gruende = []
        for d in stufe.get("deps", []):                             # jede Eingabedatei bzw. jeder Eingabeordner
            aktuell = md5_von(d["path"])
            # TODO 2.2P: wenn aktuell != d["md5"]: d["path"] an gruende anhaengen
            raise NotImplementedError("TODO 2.2P")
        for datei, werte in stufe.get("params", {}).items():        # z. B. params.yaml: {train: {...}}
            for schluessel, alt in werte.items():
                if params[schluessel] != alt:
                    gruende.append(f"{datei}: {schluessel}")
        print(f"{name:10s} {'VERALTET  <- ' + ', '.join(gruende) if gruende else 'aktuell'}")
        if gruende:
            veraltet.append(name)
    return veraltet

print("--- von Hand ---"); veraltet = status_von_hand()
print("--- DVC ---"); run("dvc status")
check("2.2P direkt nach dem Lauf: keine Stufe veraltet (wie dvc status)", lambda: veraltet == [])

### 2.3 · Parameter ändern

`DENKEN` + `CODE` (1 Zeile) · allein

1. **Erst vorhersagen** (in die Zelle unten schreiben): Welche Stufen laufen neu, wenn `lr_backbone` von 0,0001 auf 0,0002 steigt?
2. `CODE`: den Parameter mit der Hilfsfunktion `setze_param("finetune", "lr_backbone", 0.0002)` ändern (sie schreibt `params.yaml`
   neu – genauso gut könnten Sie die Datei im Editor ändern).
3. `status_von_hand()` und `dvc status` lesen, dann `dvc repro` – stimmt Ihre Vorhersage?

**Erwartung:** prepare und featurize „didn't change, skipping“; train und evaluate laufen; Val-Genauigkeit **95,4 %**.

*Meine Vorhersage:* …

In [ ]:
def setze_param(block, name, wert):
    """Setzt params.yaml -> train -> block -> name auf wert (Rest der Datei bleibt gleich)."""
    p = yaml.safe_load((ARBEIT / "params.yaml").read_text())
    p["train"][block][name] = wert
    kopf = "# Hyperparameter des Traegermodells (Werte aus Lab 2, Aufgaben 4.3-4.5)\n"
    (ARBEIT / "params.yaml").write_text(kopf + yaml.safe_dump(p, sort_keys=False), newline="\n")
    print(f"params.yaml: train.{block}.{name} = {wert}")

# TODO 2.3: lr_backbone auf 0.0002 setzen: Nutzen Sie die Funktion setze_param()
raise NotImplementedError("TODO 2.3")
print("--- von Hand ---"); veraltet = status_von_hand()
print("--- DVC ---"); run("dvc status")
erg_repro2 = run("dvc repro")
run("git add params.yaml dvc.lock lab06/reports")
run('git commit -q -m "Pipeline v2: lr_backbone 0.0002"', erwarte_fehler=True)
V2 = run("git rev-parse --short HEAD", zeigen=False).ausgabe.strip()
run("dvc push")
metriken2 = json.loads((ARBEIT / "lab06/reports/metrics.json").read_text())
gelaufen2 = [s for s in stufen if f"Running stage '{s}'" in erg_repro2.ausgabe]
print(f"\nneu gelaufen: {gelaufen2} | Val-Genauigkeit {metriken2['accuracy']:.1%}")
check("2.3 von Hand vorher erkannt: nur train veraltet (Parameter)", lambda: veraltet == ["train"])
check("2.3 dvc repro: nur train und evaluate gelaufen", lambda: gelaufen2 == ["train", "evaluate"])

**`DENKEN`:** `status_von_hand()` (und `dvc status`) meldeten vor dem Lauf nur `train` als veraltet – trotzdem lief auch
`evaluate`. Warum?

### 2.4 · Vergleichen und zurück

`CODE` (1 Zeile) · allein

1. `dvc params diff` und `dvc metrics diff` vergleichen den Arbeitsstand mit einem früheren Commit – ohne etwas auszuchecken.
   Hier: Stand v1 (`V1`) gegen jetzt.
2. `CODE`: `lr_backbone` wieder auf 0,0001 setzen; dann `dvc repro` (vorgegeben) und auf die Zeit achten.

**Erwartung:** Genauigkeit −0,03 (0,984 → 0,954), Kettensäge leidet am meisten (Recall 0,97 → 0,84); nach dem Zurücksetzen:
„Stage 'train' is cached – skipping run, checking out outputs“ – **in unter einer Sekunde**, ohne zu rechnen.

In [ ]:
run(["dvc", "params", "diff", V1])
run(["dvc", "metrics", "diff", V1])
# TODO 2.4: lr_backbone zurueck auf 0.0001
raise NotImplementedError("TODO 2.4")
erg_repro3 = run("dvc repro")
print(f"\ndvc repro dauerte {erg_repro3.sekunden:.1f} s (erster Lauf: {erg_repro1.sekunden:.0f} s)")
run("git add params.yaml dvc.lock lab06/reports")
run('git commit -q -m "Pipeline v3: zurueck auf lr_backbone 0.0001"', erwarte_fehler=True)
metriken3 = json.loads((ARBEIT / "lab06/reports/metrics.json").read_text())
check("2.4 train kam aus dem Run-Cache", lambda: "Stage 'train' is cached" in erg_repro3.ausgabe)
check(f"2.4 wieder dieselbe Genauigkeit wie v1 ({metriken3['accuracy']:.1%})", lambda: metriken3 == metriken1)

**`DENKEN` – Interpretation**

a) Warum lief `featurize` in 2.3 nicht?

b) Warum war 2.4 so schnell? Woher kamen Modell und Metriken?

c) Sie wollen statt ResNet-18 ein anderes Backbone testen (neue Gewichtsdatei). Welche Stufen laufen dann neu?

d) Modell v2 ist schlechter als v1. Mit welchen zwei Befehlen holen Sie den Stand v1 samt Modell in den Arbeitsordner zurück?

## Aufgabe 3 · Datenqualitäts- und Bias-Check

**Block nach „Beispiel: Helligkeit und Farbe je Klasse“ · 20 Min. (11 allein + 9 gemeinsam)** · Folien Abschnitt 6 · Huyen (2022) Kap. 4

Die Stufe `prepare` hat bei jedem Lauf einen Prüfbericht geschrieben. Sie lesen ihn, rechnen die Duplikatsuche von Hand nach
und vergleichen die Bilder je Klasse nach Helligkeit und Farbe – gleiche Klassengrößen schützen nicht vor Bias.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 3.1 | Datencheck lesen, Duplikate von Hand | `AUSFÜHREN` + `PRINZIP` + `CODE` · gemeinsam | 5 Min. |
| 3.2 | Helligkeit und Farbe je Klasse | `PRINZIP` + `CODE` · allein | 7 Min. |
| 3.3 | Bilder ansehen | `AUSFÜHREN` · allein | 4 Min. |
| 3.4 | Interpretation | `DENKEN` · Besprechung | 4 Min. |

### 3.1 · Datencheck lesen

`AUSFÜHREN` + `PRINZIP` + `CODE` (1 Zeile) · gemeinsam · `dvc metrics show` zeigt alle Metrik-Dateien der Pipeline,
darunter `datencheck.json`: Anzahl Bilder, defekte Dateien, Duplikate (gleiche Prüfsumme), Bildgrößen, Klassen je Split.

Danach suchen wir die Duplikate selbst: Wir gehen das Manifest Zeile für Zeile durch und sammeln für jede Prüfsumme die
Pfade, die sie haben. Eine Prüfsumme mit mehr als einem Pfad ist ein Duplikat.

**Erwartung:** 3 001 Bilder, 0 defekt, alle 160 × 160 px – und **1 Duplikat**. Welches?

In [ ]:
abschnitt("Aufgabe 3")
run("dvc metrics show")
datencheck = json.loads((ARBEIT / "lab06/reports/datencheck.json").read_text())
print(json.dumps(datencheck, indent=1))

# PRINZIP: Duplikate ueber Pruefsummen finden - eine Liste von Pfaden je md5
gruppen = {}                                                    # md5 -> [pfad, pfad, ...]
for zeile in (ARBEIT / "lab06/data/manifest.csv").read_text().splitlines()[1:]:   # [1:]: Kopfzeile ueberspringen
    split, pfad, label, md5 = zeile.split(",")
    # TODO 3.1: pfad an die Liste von md5 anhaengen
    raise NotImplementedError("TODO 3.1")
doppelt = {m: p for m, p in gruppen.items() if len(p) > 1}
print(f"\n{len(gruppen)} verschiedene Inhalte bei {sum(len(p) for p in gruppen.values())} Bildern; Duplikate:")
for m, pfade in doppelt.items():
    print(f"  {m[:12]}...: {pfade}")
check("3.1 Datencheck: 3 001 Bilder, 0 defekt, alle 160x160",
      lambda: datencheck["n"] == 3001 and datencheck["defekt"] == 0 and datencheck["groessen"] == {"160x160": 3001})
check("3.1 von Hand gefunden: genau 1 Duplikat == Datencheck", lambda: len(doppelt) == 1 == datencheck["duplikate"])

**`DENKEN` (1 Min.):** Welche Prüfungen aus dem Gruppenvortrag *Datenqualität* stecken in `datencheck.json` – welche fehlen?
Was wäre, wenn die Kopie nicht aus dem Validierungs-, sondern aus dem **Trainings**ordner stammte?

### 3.2 · Helligkeit und Farbe je Klasse

`PRINZIP` + `CODE` (1 Zeile) · allein · Die **Helligkeit** eines Pixels mit den Farbwerten $(R, G, B)$ (je 0–255) ist nach der
Norm ITU-R BT.601, die auch PIL bei `convert("L")` verwendet:

$$L = 0{,}299\,R + 0{,}587\,G + 0{,}114\,B$$

Grün zählt am stärksten, weil das Auge dafür am empfindlichsten ist. Die mittlere Helligkeit eines Bildes ist der Mittelwert
von $L$ über alle Pixel, der mittlere Blauwert der Mittelwert von $B$.

**Erst von Hand** (Selbst-rechnen-Kasten der Folie): vier Pixel $(120, 170, 230)$, $(110, 160, 220)$, $(100, 150, 210)$,
$(100, 100, 100)$. Dann für alle 2 500 Trainingsbilder je Klasse.

**Erwartung:** vier Pixel: Blau 190,0, Helligkeit 138,9 (PIL rundet je Pixel und liefert 139) – das Bild ähnelt am ehesten einem Fallschirm; je Klasse (Train): Fallschirm hell 140,1 / blau 160,1 – deutlich über
allen anderen; Kirche hell 104,8 / blau 100,3.

In [ ]:
# (a) Von Hand: vier Pixel
pixel = [(120, 170, 230), (110, 160, 220), (100, 150, 210), (100, 100, 100)]
summe_L, summe_B = 0.0, 0.0
for R, G, B in pixel:
    # TODO 3.2: Helligkeit des Pixels nach BT.601: L = ?
    raise NotImplementedError("TODO 3.2")
    print(f"Pixel ({R:3d}, {G:3d}, {B:3d}): L = {L:6.2f}")
    summe_L += L; summe_B += B
print(f"Mittel: Helligkeit {summe_L / len(pixel):.1f}, Blau {summe_B / len(pixel):.1f}")
check("3.2 vier Pixel: Helligkeit 138,9, Blau 190,0", lambda: nah(summe_L / 4, 138.92, 0.01) and summe_B / 4 == 190.0)

# Gegenprobe mit PIL: dieselbe Formel, nur mit ganzzahliger Rundung je Pixel
mini = Image.new("RGB", (4, 1)); mini.putdata(pixel)
print("PIL convert('L'):", list(mini.convert("L").getdata()))

In [ ]:
# (b) Alle Trainingsbilder: mittlere Helligkeit und mittleres Blau je Bild, dann Mittelwert je Klasse
img = ARBEIT / "data/imagenette5"
KLASSEN = [c["name"] for c in json.loads((img / "classes.json").read_text())]
zeilen = []
t0 = time.perf_counter()
for line in (img / "train.txt").read_text().splitlines():
    rel, y = line.split("\t")
    with Image.open(img / rel) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=float)          # Form (160, 160, 3)
    hell = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]).mean()   # Formel von oben, fuer alle Pixel
    blau = rgb[..., 2].mean()
    zeilen.append({"klasse": KLASSEN[int(y)], "datei": rel, "hell": hell, "blau": blau})
bilder = pd.DataFrame(zeilen)
print(f"{len(bilder)} Bilder in {time.perf_counter() - t0:.1f} s\n")
print(bilder["klasse"].value_counts().to_string(), "\n")
tab = bilder.groupby("klasse", sort=False)[["hell", "blau"]].agg(["mean", "std"]).round(1)
display(tab)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for a, spalte in zip(ax, ["hell", "blau"]):
    m = bilder.groupby("klasse", sort=False)[spalte].mean()
    a.bar(m.index, m.values, color=["#E87722" if k == "Fallschirm" else "#1F3864" for k in m.index])
    a.axhline(bilder[spalte].mean(), color="grey", ls="--", lw=1, label="alle Klassen")
    a.set_title(f"mittlere {'Helligkeit' if spalte == 'hell' else 'Blaukanal'} je Klasse (Train)", fontsize=10)
    a.tick_params(axis="x", rotation=20, labelsize=8); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
m = bilder.groupby("klasse")[["hell", "blau"]].mean()
check("3.2 je Klasse 500 Trainingsbilder", lambda: set(bilder["klasse"].value_counts()) == {500})
check(f"3.2 Fallschirm: hell {m.loc['Fallschirm', 'hell']:.1f} (140,1), blau {m.loc['Fallschirm', 'blau']:.1f} (160,1)",
      lambda: nah(m.loc["Fallschirm", "hell"], 140.1, 0.3) and nah(m.loc["Fallschirm", "blau"], 160.1, 0.3))
check("3.2 Fallschirm ist die hellste und blaueste Klasse",
      lambda: m["hell"].idxmax() == "Fallschirm" and m["blau"].idxmax() == "Fallschirm")

### 3.3 · Bilder ansehen

`AUSFÜHREN` · allein · Zahlen allein beweisen nichts. Die Funktion zeigt für eine Klasse die 12 hellsten und die 12 dunkelsten
Trainingsbilder. Schauen Sie sich *Fallschirm* und *Kirche* an – und dann eine Klasse Ihrer Wahl (`KLASSEN` zeigt die Namen).

**Erwartung:** helle Fallschirme vor blauem Himmel; dunkle Kirchen sind oft Innenräume oder Nachtaufnahmen.

In [ ]:
def zeige_extreme(klasse, n=12):
    """Obere Reihe: die n hellsten Bilder der Klasse, untere Reihe: die n dunkelsten."""
    teil = bilder[bilder["klasse"] == klasse].sort_values("hell")
    fig, ax = plt.subplots(2, n, figsize=(1.25 * n, 3.0))
    for reihe, auswahl, titel in [(0, teil.tail(n)[::-1], "hellste"), (1, teil.head(n), "dunkelste")]:
        for a, (_, r) in zip(ax[reihe], auswahl.iterrows()):
            a.imshow(Image.open(img / r["datei"])); a.set_xticks([]); a.set_yticks([])
            a.set_title(f"{r['hell']:.0f}", fontsize=7)
        ax[reihe, 0].set_ylabel(titel, fontsize=8)
    fig.suptitle(f"{klasse}: {n} hellste (oben) und {n} dunkelste (unten) Trainingsbilder – Zahl = mittlere Helligkeit", fontsize=9)
    plt.tight_layout(); plt.show()

print(KLASSEN)
zeige_extreme("Fallschirm")
zeige_extreme("Kirche")
# zeige_extreme("Golfball")      # eigene Wahl

### 3.4 · Interpretation

`DENKEN` · Besprechung (4 Min.)

a) Ist „Fallschirm = hell und blau“ ein **Bias** des Datensatzes oder eine legitime **Eigenschaft** der Klasse? Wovon hängt die
   Antwort ab?
b) Welche Gefahr besteht für das Modell? Woran würden Sie im Einsatz merken, dass es eingetreten ist?
c) Nennen Sie zwei Gegenmaßnahmen – eine auf der Datenseite, eine beim Training.
d) Übertragen Sie die Frage auf Ihr Capstone-Projekt: Welche Bedingung (Licht, Ort, Sensor, Zeitraum, Personengruppe) fehlt
   vermutlich in Ihren Daten? Die Antwort gehört ins Datenkonzept, Feld „Bias & Repräsentativität“.

## Aufgabe 4 · Docker-Installationscheck

**Während der Capstone-Sprechstunde · 10 Min., allein** · Vorbereitung auf *MLOps: Vom Notebook zum Produktivsystem*

Im nächsten Block packen Sie das Trägermodell als Service in einen Container. Dafür muss Docker laufen. Die Zellen prüfen das in
drei Schritten:

| Teil | Befehl | Was er zeigt | Zeit |
|---|---|---|---|
| 4.1 | `docker --version` · `docker info` | Ist der Client installiert? Läuft der Docker-Dienst? | 3 Min. |
| 4.2 | `docker run --rm hello-world` | Lädt ein winziges Testimage und startet es | 3 Min. |
| 4.3 | – | Wenn es nicht läuft: Ursache notieren (Tabelle unten) | 4 Min. |

**Erwartung:** „Hello from Docker!“ in der Ausgabe von 4.2. **Ergebnis bitte melden** (läuft / läuft nicht + Ursache) –
Weg laut Lab-Briefing.

In [ ]:
abschnitt("Aufgabe 4")
DOCKER = {"client": False, "dienst": False, "hello_world": False}
if shutil.which("docker") is None:
    print("❌ Kein Programm 'docker' gefunden -> Docker Desktop ist nicht installiert (oder Jupyter vor der Installation gestartet).")
else:
    # 4.1 Client und Dienst
    DOCKER["client"] = run("docker --version", cwd=REPO_ROOT, erwarte_fehler=True).ok
    DOCKER["dienst"] = run("docker info --format {{.ServerVersion}}", cwd=REPO_ROOT, erwarte_fehler=True).ok
    # 4.2 Testcontainer (nur, wenn der Dienst laeuft; laedt beim ersten Mal ca. 10 kB aus dem Internet)
    if DOCKER["dienst"]:
        erg = run("docker run --rm hello-world", cwd=REPO_ROOT, erwarte_fehler=True)
        DOCKER["hello_world"] = "Hello from Docker!" in erg.ausgabe
print("\nErgebnis:", DOCKER)
if all(DOCKER.values()):
    print("✅ Docker laeuft - Sie sind bereit fuer MLOps.")
elif DOCKER["client"] and not DOCKER["dienst"]:
    print("⚠️  Docker ist installiert, aber der Dienst laeuft nicht: Docker Desktop starten, ~30 s warten, Zelle erneut ausfuehren.")
else:
    print("⚠️  Docker laeuft nicht - Ursache mit der Tabelle in 4.3 eingrenzen und notieren.")

### 4.3 · Wenn es nicht läuft

`DENKEN` · Ursache eingrenzen und notieren. Häufige Ursachen laut Docker-Dokumentation (Install Docker Desktop on Windows/Mac):

| Meldung / Symptom | Wahrscheinliche Ursache | Was tun |
|---|---|---|
| `docker` nicht gefunden | nicht installiert, oder Jupyter vor der Installation gestartet | Docker Desktop installieren, Jupyter neu starten |
| „Cannot connect to the Docker daemon“ / „error during connect“ | Docker Desktop läuft nicht | Docker Desktop starten, warten, bis das Wal-Symbol ruhig ist |
| Windows: „WSL 2 installation is incomplete“ / WSL zu alt | WSL fehlt oder < 2.1.5 | `wsl --update` (Admin-Terminal) |
| Windows: „Virtualization support not detected“ | Hardware-Virtualisierung im BIOS/UEFI aus | im BIOS/UEFI einschalten (Intel VT-x / AMD-V) |
| Installation verweigert | Windows-Version zu alt (Win 10 < 22H2, Win 11 < 23H2), < 8 GB RAM, fehlende Adminrechte auf verwaltetem Gerät | anderes Gerät oder Fallback |

**Fallback, wenn es gar nicht geht** (Lab-Briefing): Podman oder Rancher Desktop → Image-Build in GitHub Actions → Build durch ein
Teammitglied. Bewertet werden in MLOps Dockerfile und CI-Log, nicht der lokale Build – Sie verlieren also nichts.

*Mein Ergebnis:* läuft / läuft nicht, weil …

## Zusatz (zu Hause) · Zeitreise und ein harmloser Kommentar

### Z1 · Zeitreise: Stand v2 wiederherstellen

`AUSFÜHREN` · `git checkout <commit>` setzt die Zeiger (`.dvc`, `dvc.lock`, `params.yaml`, Metriken) zurück, `dvc checkout`
holt die passenden Daten und das passende Modell aus dem Cache in den Arbeitsordner. Danach zurück auf `main`.

**Erwartung:** Nach `git checkout` allein meldet `dvc status`, dass das Modell nicht zu den Zeigern passt; nach `dvc checkout`
hat das Modell im Arbeitsordner wieder die Prüfsumme aus v2. Zurück auf `main`: wieder v1/v3.

In [ ]:
abschnitt("Zusatz")
def md5_modell():
    return hashlib.md5((ARBEIT / "models/traeger_v1.pt").read_bytes()).hexdigest()
lock_md5 = lambda: [o["md5"] for o in yaml.safe_load((ARBEIT / "dvc.lock").read_text())["stages"]["train"]["outs"]
                    if o["path"].endswith(".pt")][0]
run("git log --oneline")
run(["git", "checkout", "-q", V2])                     # nur die Zeiger springen auf v2
print(f"Modell im Ordner {md5_modell()[:10]}... | laut dvc.lock {lock_md5()[:10]}...")
run("dvc status")
run("dvc checkout")                                    # Daten und Modell passend zu den Zeigern
md5_v2 = md5_modell()
print(f"nach dvc checkout: {md5_v2[:10]}... | Genauigkeit laut metrics.json:",
      json.loads((ARBEIT / "lab06/reports/metrics.json").read_text())["accuracy"])
ok_v2 = md5_v2 == lock_md5()
run("git checkout -q main"); run("dvc checkout")       # zurueck zum aktuellen Stand
check("Z1 nach dvc checkout passt das Modell zu v2", ok_v2)
check("Z1 zurueck auf main: Modell passt wieder zu dvc.lock", lambda: md5_modell() == lock_md5())

### Z2 · Ein Kommentar in `train.py`

`DENKEN` + `AUSFÜHREN` · Fall (e) der Folie „Was läuft nach einer Änderung neu?“: Sie fügen in `train.py` nur einen Kommentar
ein. **Vorhersage:** Läuft `train`? Läuft danach auch `evaluate`? Dann ausführen (≈ 1–2 Min.) und zum Schluss die Änderung
zurücknehmen.

**Erwartung:** `train` läuft (die Datei hat einen neuen Hash – DVC versteht keinen Code). Das neue Modell ist aber **bitgleich**
(fester Seed, gleiche Maschine, kein Datum in der Ausgabe) – deshalb wird `evaluate` übersprungen. Nach dem Zurücknehmen kommt
`train` aus dem Run-Cache.

In [ ]:
md5_vorher = md5_modell()
skript = ARBEIT / "lab06/train.py"
skript.write_text(skript.read_text() + "# Nur ein Kommentar - aendert nichts am Ergebnis\n", newline="\n")
run("dvc status")
erg_z2 = run("dvc repro")
print(f"\nModell vorher {md5_vorher[:10]}... | nachher {md5_modell()[:10]}... | bitgleich: {md5_vorher == md5_modell()}")
check("Z2 train ist gelaufen", lambda: "Running stage 'train'" in erg_z2.ausgabe)
check("Z2 Modell bitgleich -> evaluate uebersprungen",
      lambda: md5_vorher == md5_modell() and "Running stage 'evaluate'" not in erg_z2.ausgabe,
      "auf manchen Rechnern/Thread-Zahlen weichen die letzten Bits ab - dann laeuft evaluate auch (kein Fehler)")
run("git checkout -- lab06/train.py")                 # Kommentar zuruecknehmen
erg_z2b = run("dvc repro")                             # alte Kombination -> Run-Cache
run('git commit -q -am "nach Z2 aufgeraeumt"', erwarte_fehler=True)

**Weiterdenken:** Die ResNet-Gewichte (46 MB) liegen noch in Git. Wie stellen Sie sie unter DVC – und welche Stufen laufen
danach neu? (Tipp: `git rm --cached` + `dvc add`; die Prüfsumme der Datei ändert sich dadurch nicht.)

## Abgabe, Laufzeiten und Prüfungen

`AUSFÜHREN` · Die Zelle packt genau das in ein ZIP, was in Git liegt (`git ls-files`): Code, Zeiger (`.dvc`, `dvc.lock`),
`dvc.yaml`, `params.yaml` und die Metriken – **keine** Bilder, kein Cache, kein Modell. Einzige Ausnahme: Die 46 MB großen
ResNet-Gewichte liegen noch in Git (siehe „Weiterdenken“) und werden hier ausgelassen. Mehr braucht niemand, um Ihren Stand mit
Zugriff auf das Remote vollständig wiederherzustellen (`git clone` + `dvc pull` + `dvc repro`).

In [ ]:
abschnitt("Abschluss")
import zipfile
zip_pfad = (RESULTS / "abgabe_traeger_pipeline.zip").resolve()
namen = [n for n in run("git ls-files", zeigen=False).ausgabe.splitlines() if not n.endswith(".pth")]   # alles aus Git ausser Gewichten
with zipfile.ZipFile(zip_pfad, "w", zipfile.ZIP_DEFLATED) as z:
    for n in namen:
        z.write(ARBEIT / n, arcname=f"traeger-pipeline/{n}")
print(f"{zip_pfad.name}: {len(namen)} Dateien, {zip_pfad.stat().st_size / 1e3:.0f} kB")
print("\n".join(namen))

display(pd.DataFrame({"Rechenzeit (s)": RUNTIME}).round(1))
print("Langsamste Befehle:")
display(pd.Series(LAUFZEITEN, name="s").sort_values(ascending=False).head(6).round(1))
print(f"Pruefungen: {sum(CHECKS.values())} ✅ / {sum(not v for v in CHECKS.values())} ❌")
for name, ok in CHECKS.items():
    if not ok:
        print("  ❌", name)